In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

In [3]:
from dataclasses import dataclass

# 실행 컨텍스트 정의 (누가 실행하는지 식별)
@dataclass
class Context:
    user_id: str
    app_name: str

In [4]:
from typing import TypedDict

# LLM이 추출해야 할 정보의 구조를 정의
class UserInfo(TypedDict):
    personal_info: str
    preference: str

In [5]:
from langchain_core.runnables import RunnableConfig
from langchain.tools import tool, ToolRuntime
from langchain_core.tools import InjectedToolArg
from typing import Annotated

# Tool 정의: 사용자의 정보를 조회
@tool
def get_user_info(runtime: Annotated[ToolRuntime, InjectedToolArg]) -> str:
    """
    현재 사용자의 정보 조회 (시스템 내부용 도구)
    """
    user_id = runtime.context.user_id
    app = runtime.context.app_name

    print("user_id, app:",user_id, app)
    
    # 해당 네임스페이스의 모든 메모리 검색
    memories = runtime.store.search((user_id, app))

    print("memories:",memories)

    if not memories:
        return "기록된 정보 없음"

    results = []
    for item in memories:
        # 저장된 데이터 구조(UserInfo)에 맞춰 필드 확인
        data = item.value
        # 저장할 때 사용한 Key들을 확인해서 문자열로 변환
        if "personal_info" in data:
            results.append(f"- 개인정보: {data['personal_info']}")
        if "preference" in data:
            results.append(f"- 선호도: {data['preference']}")

    return "\n".join(results) if results else "데이터 형식 불일치로 읽을 수 없음"

In [6]:
import uuid

@tool
def save_user_info(user_info: UserInfo, runtime: Annotated[ToolRuntime, InjectedToolArg]):
    """
    사용자의 정보를 저장하거나 업데이트
    """
    # 1. 실행 컨텍스트에서 user_id 가져오기
    user_id = runtime.context.user_id
    app = runtime.context.app_name
    store = runtime.store

    print("store:",store)

    # 2. Store에 데이터 저장 (put(namespace, key, value))
    memory_key = str(uuid.uuid4())
    store.put((user_id, app),memory_key, user_info)

    return f"정보가 안전하게 저장되었습니다. (ID: {memory_key})"

In [7]:
system_message = "당신은 사용자의 정보를 기억하는 비서입니다. 사용자가 자신의 정보를 말하면 반드시 'save_user_info' 도구를 사용하여 저장하세요."

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[get_user_info, save_user_info], 
    store=store,
    context_schema=Context,
    system_prompt=system_message
)

In [9]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 이제 '김일남'이야"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)

c:\Users\dandycode\Documents\GitHub\kor-it-langchain-class\ch10\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


store: <langgraph.store.memory.InMemoryStore object at 0x00000226AB5E2120>


In [11]:
response

{'messages': [HumanMessage(content="내 이름은 이제 '김일남'이야", additional_kwargs={}, response_metadata={}, id='c4f7e91d-7255-446d-8f23-d52139106e0d'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'save_user_info', 'arguments': '{"user_info": {"personal_info": "\\uc774\\ub984: \\uae40\\uc77c\\ub0a8", "preference": "\\uc5c6\\uc74c"}}'}, '__gemini_function_call_thought_signatures__': {'ca89e8e0-2b38-48ab-b8c8-171ad9e77b37': 'EjQKMgEMOdbHPM10jNk9pXDRn3Ir5i3WLvHBjmWRZ6SRAxpXe+lNFs0cCZfuUm3HP1O+4RF8'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7adb-2ed5-7772-9981-860409d13a30-0', tool_calls=[{'name': 'save_user_info', 'args': {'user_info': {'personal_info': '이름: 김일남', 'preference': '없음'}}, 'id': 'ca89e8e0-2b38-48ab-b8c8-171ad9e77b37', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 151, 'output_tokens': 36, 'total_tokens': 187, 'inpu

In [12]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나는 아이스 아메리카노를 좋아해"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)

c:\Users\dandycode\Documents\GitHub\kor-it-langchain-class\ch10\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


store: <langgraph.store.memory.InMemoryStore object at 0x00000226AB5E2120>


In [13]:
response

{'messages': [HumanMessage(content='나는 아이스 아메리카노를 좋아해', additional_kwargs={}, response_metadata={}, id='0a5a3635-3f4d-4153-a72b-1300f9cc1a66'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'save_user_info', 'arguments': '{"user_info": {"preference": "\\uc544\\uc774\\uc2a4 \\uc544\\uba54\\ub9ac\\uce74\\ub178\\ub97c \\uc88b\\uc544\\ud568", "personal_info": "\\uc5c6\\uc74c"}}'}, '__gemini_function_call_thought_signatures__': {'500d12d2-de34-4fb7-97ba-16155b2fc74c': 'EjQKMgEMOdbHTmNLzE1XgIKIm+xMJOnqoqjpE5vc4Bd0EiEZhIt8dxLxEH19lxzatN6nyKFy'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7adb-dd0b-7ae2-b949-80e550d6470f-0', tool_calls=[{'name': 'save_user_info', 'args': {'user_info': {'preference': '아이스 아메리카노를 좋아함', 'personal_info': '없음'}}, 'id': '500d12d2-de34-4fb7-97ba-16155b2fc74c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_toke

In [14]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나는 랭체인 공부를 좋아해. 학습용 챗봇 프로젝트를 진행 중이야."}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)

c:\Users\dandycode\Documents\GitHub\kor-it-langchain-class\ch10\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


store: <langgraph.store.memory.InMemoryStore object at 0x00000226AB5E2120>


In [15]:
response

{'messages': [HumanMessage(content='나는 랭체인 공부를 좋아해. 학습용 챗봇 프로젝트를 진행 중이야.', additional_kwargs={}, response_metadata={}, id='7ac0e2ca-721d-4b75-9e5a-214b3c5e11b2'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'save_user_info', 'arguments': '{"user_info": {"preference": "\\ub7ad\\uccb4\\uc778 \\ubc0f \\ucc57\\ubd07 \\uac1c\\ubc1c \\uad00\\ub828 \\uad00\\uc2ec\\uc0ac.", "personal_info": "\\ub7ad\\uccb4\\uc778 \\uacf5\\ubd80\\ub97c \\uc88b\\uc544\\ud558\\uba70, \\ud559\\uc2b5\\uc6a9 \\ucc57\\ubd07 \\ud504\\ub85c\\uc81d\\ud2b8\\ub97c \\uc9c4\\ud589 \\uc911\\uc784."}}'}, '__gemini_function_call_thought_signatures__': {'b1c65d33-a5f2-4d44-a773-951b750db4b3': 'EjQKMgEMOdbHSD+2nI0beanH75k5XU3xjpKtoY+KxXi/Ys2h3n+VZfZO3EL/owYF+4O/tRo5'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7adc-7461-7532-aaf9-5ea55d7de172-0', tool_calls=[{'name': 'save_user_info', 'args'

In [16]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 알고 있는 모든 것을 말해줘"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)

c:\Users\dandycode\Documents\GitHub\kor-it-langchain-class\ch10\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


user_id, app: user_001 personal_assistant
memories: [Item(namespace=['user_001', 'personal_assistant'], key='914ba760-215d-4aaa-a8d8-289bc52da1d4', value={'personal_info': '이름: 김일남', 'preference': '없음'}, created_at='2026-05-30T21:47:31.572463+00:00', updated_at='2026-05-30T21:47:31.572463+00:00', score=None), Item(namespace=['user_001', 'personal_assistant'], key='1d2b8417-4140-408b-a7e1-61776d629730', value={'personal_info': '없음', 'preference': '아이스 아메리카노를 좋아함'}, created_at='2026-05-30T21:48:15.451697+00:00', updated_at='2026-05-30T21:48:15.451697+00:00', score=None), Item(namespace=['user_001', 'personal_assistant'], key='d6a33a44-9e8c-4cdb-87f5-71837ed59f6f', value={'personal_info': '랭체인 공부를 좋아하며, 학습용 챗봇 프로젝트를 진행 중임.', 'preference': '랭체인 및 챗봇 개발 관련 관심사.'}, created_at='2026-05-30T21:48:52.177640+00:00', updated_at='2026-05-30T21:48:52.177640+00:00', score=None)]


In [17]:
response

{'messages': [HumanMessage(content='나에 대해 알고 있는 모든 것을 말해줘', additional_kwargs={}, response_metadata={}, id='ddde2ed8-c49a-4231-b6d7-63afe7a760bb'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_user_info', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'a77b4a4a-7775-4956-af97-e73de635c017': 'EjQKMgEMOdbHDXp9l8G9vC/4hVZ6Ki6OcMCUJz/3EgClnq+LoM3dQGIxviyLRyjnh/KTINu6'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7adc-cf17-74c1-b1bd-c763d581886a-0', tool_calls=[{'name': 'get_user_info', 'args': {}, 'id': 'a77b4a4a-7775-4956-af97-e73de635c017', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 151, 'output_tokens': 12, 'total_tokens': 163, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='- 개인정보: 이름: 김일남\n- 선호도: 없음\n- 개인정보: 없음\n- 선호도: 아이스 아메리카노를 좋아함\n- 개인정보: 랭체인 공부를 좋아하며, 학습용 챗봇 프로젝트를 진